## Scraping

In [6]:
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
OUTPUT_FILE = "books.csv"

books = []

for page in range(1, 6):
    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}catalogue/page-{page}.html"

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):
        # Title
        title = book.select_one("h3 a")["title"]

        # Price: "£51.77" -> 51.77
        price_text = book.select_one(".price_color").get_text(strip=True)
        price = float(price_text.replace("Â£", ""))# remove the currency sign

        # Rating: "Three" -> 3
        rating_words = {
            "One": 1,
            "Two": 2,
            "Three": 3,
            "Four": 4,
            "Five": 5,
        }

        rating_class = book.select_one(".star-rating")["class"][1]
        rating = rating_words[rating_class]

        # Stock: "In stock" -> True
        availability = book.select_one(".availability").get_text(" ", strip=True)
        in_stock = "In stock" in availability

        # Relative URL -> full URL
        relative_url = book.select_one("h3 a")["href"]
        book_url = urljoin(url, relative_url)

        books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "in_stock": in_stock,
            "url": book_url,
        })

# Save exactly the requested columns
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["title", "price", "rating", "in_stock", "url"]
    )

    writer.writeheader()
    writer.writerows(books)

print(f"Saved {len(books)} books to {OUTPUT_FILE}")

Saved 100 books to books.csv


## SQL

SQLite was used because it is sufficient for the scope of this task. Using MySQL or SQL Server would require additional server or local database setup, which would add unnecessary complexity without providing any benefit for this task.

### Add data

In [7]:
import csv
import sqlite3

CSV_FILE = "books.csv"
DB_FILE = "books.db"

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        title TEXT,
        price REAL,
        rating INTEGER,
        in_stock BOOLEAN,
        url TEXT
    )
""")

with open(CSV_FILE, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    rows = [
        (
            row["title"],
            float(row["price"]),
            int(row["rating"]),
            row["in_stock"].lower() == "true",
            row["url"]
        )
        for row in reader
    ]

cursor.executemany("""
    INSERT INTO books (title, price, rating, in_stock, url)
    VALUES (?, ?, ?, ?, ?)
""", rows)

conn.commit()
conn.close()

print(f"Loaded {len(rows)} books into {DB_FILE}")

Loaded 100 books into books.db


### Answer questions

In [9]:
#  1. Average price for each rating
import csv
import sqlite3

CSV_FILE = "books.csv"
DB_FILE = "books.db"

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
    SELECT rating , AVG(price) FROM books GROUP BY rating
""")

cursor.fetchall()


[(1, 35.51954545454545),
 (2, 35.909473684210525),
 (3, 36.83909090909091),
 (4, 33.98166666666666),
 (5, 30.012105263157896)]

In [14]:
# 2. The 5 most expensive books rated 4 or 5
import csv
import sqlite3

CSV_FILE = "books.csv"
DB_FILE = "books.db"

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
    SELECT title,rating,price FROM books Where rating > 3 ORDER BY price DESC LIMIT 5
""")

cursor.fetchall()


[('The Death of Humanity: and the Case for Life', 4, 58.11),
 ('The Past Never Ends', 4, 56.5),
 ('Sapiens: A Brief History of Humankind', 5, 54.23),
 ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 52.29),
 ('Behind Closed Doors', 4, 52.22)]

In [22]:
# 3. How many books are out of stock, per rating

import csv
import sqlite3

CSV_FILE = "books.csv"
DB_FILE = "books.db"

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
    SELECT rating, COUNT(*) AS out_of_stock FROM books Where in_stock =  false GROUP BY rating
""")

cursor.fetchall()


[]

The result of the last SQL query is an empty list ([]), which means that no books matched the condition in_stock = false.

I checked the website again, and the first 100 books are all listed as In stock. Therefore, there are no out-of-stock books among the first 100 books, so the empty result is expected.